# Consolidación y depuración del dataset de noticias

Este notebook consolida la revisión de calidad y la depuración del corpus de noticias políticas. El propósito es obtener una base consistente para las etapas posteriores de procesamiento de lenguaje natural, procurando eliminar solo registros claramente incompletos o redundantes.


In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

## 1. Carga e inspección inicial

Se parte del archivo unificado de noticias y se realiza una primera revisión de su estructura. Las siguientes salidas permiten reconocer las dimensiones del dataset, las columnas disponibles, la distribución de registros por medio y la presencia de valores faltantes antes de aplicar cualquier criterio de limpieza.


In [2]:
df = pd.read_csv("noticias_completas_unificado.csv")

df.head()

,medio,fecha,titulo,autor,url,resumen,tags,texto_completo
0,Canal N,2026-07-22,TC declaró fundado el habeas corpus de Nicanor...,Alvaro Vega Lavado,https://canaln.pe/actualidad/tc-declaro-fundad...,El supremo tribunal ordenó remitir copias a la...,"nicanor boluarte, TC",El Tribunal Constitucional (TC) declaró fundad...
1,Canal N,2026-07-22,"Abogado de Vladimir Cerrón: ""El TC le manda un...",Alvaro Vega Lavado,https://canaln.pe/actualidad/abogado-vladimir-...,El abogado del exgobernador y líder de Perú Li...,"Vladimir Cerrón, Humberto Abanto","Humberto Abanto , abogado defensor del procesa..."
2,Canal N,2026-07-22,Senamhi: altas temperaturas seguirán hasta el ...,Alvaro Vega Lavado,https://canaln.pe/actualidad/senamhi-altas-tem...,El organismo meteorológico proyecta valores de...,Senamhi,El Servicio Nacional de Meteorología e Hidrolo...
3,Canal N,2026-07-22,Martín Vizcarra seguirá en prisión: PJ rechazó...,Alvaro Vega Lavado,https://canaln.pe/actualidad/martin-vizcarra-s...,El fallo judicial confirmó que el expresidente...,Martín Vizcarra,El procesado expresidente Martín Vizcarra Corn...
4,Canal N,2026-07-22,TC declaró nula la prisión preventiva de Vladi...,Alvaro Vega Lavado,https://canaln.pe/actualidad/tc-declaro-nula-p...,El supremo tribunal argumentó deficiencias en ...,"Vladimir Cerrón, TC",El Tribunal Constitucional (TC) declaró nula l...


In [3]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

print("\nColumnas:")
print(df.columns.tolist())

Filas: 41823
Columnas: 8

Columnas:
['medio', 'fecha', 'titulo', 'autor', 'url', 'resumen', 'tags', 'texto_completo']


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41823 entries, 0 to 41822
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   medio           41823 non-null  str  
 1   fecha           41822 non-null  str  
 2   titulo          41822 non-null  str  
 3   autor           31416 non-null  str  
 4   url             41823 non-null  str  
 5   resumen         41779 non-null  str  
 6   tags            31171 non-null  str  
 7   texto_completo  41319 non-null  str  
dtypes: str(8)
memory usage: 2.6 MB


In [5]:
df["medio"].value_counts(dropna=False)

medio
Canal N         11786
El Peruano      10217
RPP              9626
Perú21           8616
La República     1197
El Comercio       381
Name: count, dtype: int64

In [6]:
df.isnull().sum()

medio                 0
fecha                 1
titulo                1
autor             10407
url                   0
resumen              44
tags              10652
texto_completo      504
dtype: int64

## 2. Revisión de calidad

La evaluación inicial busca posibles problemas de calidad: URLs duplicadas, títulos repetidos, noticias sin cuerpo y textos exactamente iguales. Estos indicadores se examinan por separado porque una coincidencia en el título o en la URL no siempre implica que todo el registro sea redundante.


In [7]:
print("URLs duplicadas:", df.duplicated(subset=["url"]).sum())

URLs duplicadas: 0


In [8]:
print("Títulos duplicados:", df.duplicated(subset=["titulo"]).sum())

Títulos duplicados: 77


In [9]:
sin_texto = df[
    df["texto_completo"].isna() |
    (df["texto_completo"].fillna("").str.strip() == "")
]

print("Noticias sin texto completo:", len(sin_texto))

sin_texto["medio"].value_counts(dropna=False)

Noticias sin texto completo: 504


medio
El Peruano      193
Canal N         133
Perú21          103
El Comercio      47
La República     26
RPP               2
Name: count, dtype: int64

In [10]:
df_con_texto = df[
    df["texto_completo"].notna() &
    (df["texto_completo"].str.strip() != "")
].copy()

print("Noticias con texto válido:", len(df_con_texto))

Noticias con texto válido: 41319


In [11]:
duplicados_texto = df_con_texto[
    df_con_texto.duplicated(
        subset=["texto_completo"],
        keep=False
    )
]

print("Noticias con texto repetido:", len(duplicados_texto))

Noticias con texto repetido: 22


In [12]:
duplicados_texto[
    ["medio", "fecha", "titulo", "url", "texto_completo"]
].sort_values("texto_completo").head(30)

,medio,fecha,titulo,url,texto_completo
23289,La República,2026-04-16 16:41:32,Cuánto se paga de multa por no votar en Elecci...,https://larepublica.pe/politica/2026/04/16/mul...,"Al igual que en la primera vuelta, votar en la..."
22703,La República,2026-06-02 17:47:32,Segunda vuelta de las Elecciones 2026: conoce ...,https://larepublica.pe/politica/2026/06/02/seg...,"Al igual que en la primera vuelta, votar en la..."
22792,La República,2026-05-21 11:20:08,Cuánto pagaré de multa si no voto el 7 de juni...,https://larepublica.pe/politica/2026/05/21/mul...,"Al igual que en la primera vuelta, votar en la..."
22996,La República,2026-05-22 21:31:07,"Debate presidencial: JNE define la fecha, form...",https://larepublica.pe/politica/2026/05/22/deb...,El Jurado Nacional de Elecciones (JNE) confirm...
22995,La República,2026-05-20 08:43:39,"Keiko Fujimori vs Roberto Sánchez: fecha, form...",https://larepublica.pe/politica/2026/05/20/deb...,El Jurado Nacional de Elecciones (JNE) confirm...
22683,La República,2026-06-07 12:43:10,"Sánchez en su desayuno familiar en Huaral: ""I...",https://larepublica.pe/verificador/2026/06/07/...,"En el marco de la segunda vuelta electoral, el..."
22687,La República,2026-06-07 13:19:01,"Sánchez en su desayuno familiar en Huaral: ""I...",https://larepublica.pe/politica/2026/06/07/rob...,"En el marco de la segunda vuelta electoral, el..."
22436,La República,2026-06-13 16:02:33,Resultados ONPE EN VIVO: cifras actualizadas y...,https://larepublica.pe/politica/2026/06/13/res...,En esta transmisión en vivo te mostramos el av...
22472,La República,2026-06-09 20:20:28,Resultados ONPE EN VIVO HOY: sigue el conteo o...,https://larepublica.pe/politica/2026/06/09/res...,En esta transmisión en vivo te mostramos el av...
22473,La República,2026-06-07 15:37:37,Resultados oficiales ONPE EN VIVO: conteo de v...,https://larepublica.pe/politica/2026/06/07/res...,En esta transmisión en vivo te mostramos el av...


In [13]:
titulos_repetidos = df[
    df.duplicated(subset=["titulo"], keep=False)
].sort_values("titulo")

titulos_repetidos[
    ["medio", "fecha", "titulo", "url"]
].head(30)

,medio,fecha,titulo,url
14008,El Peruano,2026-02-24 00:00:00,Aldo Prieto Barrera continúa como titular del ...,https://elperuano.pe/noticia/289889-aldo-priet...
13635,El Peruano,2026-03-17 00:00:00,Aldo Prieto Barrera continúa como titular del ...,https://elperuano.pe/noticia/291469-aldo-priet...
1453,Canal N,2026-04-24,Allanan vivienda de Piero Corvetto y otros fun...,https://canaln.pe/actualidad/allanan-vivienda-...
23482,La República,2026-04-24 06:48:47,Allanan vivienda de Piero Corvetto y otros fun...,https://larepublica.pe/politica/2026/04/24/all...
1477,Canal N,2026-04-22,Amadeo Flores Carcagno jura como nuevo ministr...,https://canaln.pe/actualidad/amadeo-flores-car...
13035,El Peruano,2026-04-22 00:00:00,Amadeo Flores Carcagno jura como nuevo ministr...,https://elperuano.pe/noticia/294135-amadeo-flo...
25580,Perú21,2026-01-13 15:41:00,Archivan caso contra Guido Bellido por delito ...,https://peru21.pe/politica/archivan-caso-contr...
3171,Canal N,2026-01-13,Archivan caso contra Guido Bellido por delito ...,https://canaln.pe/actualidad/archivan-caso-con...
18363,El Peruano,2025-03-29 00:00:00,Articulan acciones contra el crimen,https://elperuano.pe/noticia/267139-articulan-...
17184,El Peruano,2025-06-26 00:00:00,Articulan acciones contra el crimen,https://elperuano.pe/noticia/273344-articulan-...


## 3. Construcción del dataset limpio inicial

En esta primera depuración se eliminan únicamente las noticias que no contienen cuerpo y los duplicados exactos del texto. Cuando un contenido aparece más de una vez, se conserva su primera publicación según la fecha; así se obtiene una base inicial limpia sin aplicar todavía criterios de similitud aproximada.


In [14]:
# Trabajamos sobre una copia
df_limpio = df.copy()

# Convertir fecha para poder ordenar correctamente
df_limpio["fecha"] = pd.to_datetime(
    df_limpio["fecha"],
    format="mixed",
    errors="coerce",
    utc=True
)

# Eliminar noticias sin texto completo
df_limpio = df_limpio[
    df_limpio["texto_completo"].notna() &
    (df_limpio["texto_completo"].str.strip() != "")
].copy()

# Ordenar de la más antigua a la más reciente
df_limpio = df_limpio.sort_values("fecha")

# Mantener solamente la primera aparición de cada texto
df_limpio = df_limpio.drop_duplicates(
    subset=["texto_completo"],
    keep="first"
)

# Restaurar índice
df_limpio = df_limpio.reset_index(drop=True)

print("Noticias originales:", len(df))
print("Noticias limpias:", len(df_limpio))
print("Registros eliminados:", len(df) - len(df_limpio))

Noticias originales: 41823
Noticias limpias: 41307
Registros eliminados: 516


In [15]:
print(
    "Textos vacíos:",
    df_limpio["texto_completo"].isna().sum()
)

print(
    "Textos duplicados:",
    df_limpio.duplicated(subset=["texto_completo"]).sum()
)

print("\nNoticias por medio:")
print(df_limpio["medio"].value_counts())

Textos vacíos: 0
Textos duplicados: 0

Noticias por medio:
medio
Canal N         11653
El Peruano      10023
RPP              9624
Perú21           8513
La República     1162
El Comercio       332
Name: count, dtype: int64


## 4. Detección exploratoria de cuasi-duplicados

Para identificar textos muy parecidos, cada noticia se representa mediante TF-IDF, una técnica que asigna mayor importancia a los términos característicos de un documento y reduce el peso de los más frecuentes. La cercanía entre estas representaciones se mide con similitud coseno: cuanto más próximo a 1 es el valor, mayor es el parecido textual.

Antes de extender el procedimiento a toda la colección, una prueba sobre una muestra permite comprobar que la representación y los umbrales produzcan candidatos razonables con un costo computacional menor. Esta exploración sirve para ajustar el enfoque, pero no determina por sí sola qué noticias deben eliminarse.

## 5. Detección sobre el corpus completo

A continuación se vectoriza el corpus limpio completo. Se utiliza `NearestNeighbors` para localizar, en cada caso, los textos más cercanos sin construir una matriz de similitud entre todos los pares, que sería demasiado grande y costosa para este volumen de noticias.


Vectorizar todo el corpus

In [16]:
vectorizador = TfidfVectorizer(
    lowercase=True,
    max_features=20000
)

tfidf_total = vectorizador.fit_transform(
    df_limpio["texto_completo"]
)

print(tfidf_total.shape)

(41307, 20000)


In [17]:
from sklearn.neighbors import NearestNeighbors

modelo_vecinos = NearestNeighbors(
    n_neighbors=3,
    metric="cosine",
    algorithm="brute"
)

modelo_vecinos.fit(tfidf_total)

distancias, indices = modelo_vecinos.kneighbors(tfidf_total)

In [18]:
pares_candidatos = []

for i in range(len(df_limpio)):

    for posicion in range(1, indices.shape[1]):
        j = indices[i, posicion]

        if j != i:
            similitud = 1 - distancias[i, posicion]

            if similitud >= 0.95:
                pares_candidatos.append((i, j, similitud))

            break

print("Pares candidatos:", len(pares_candidatos))

Pares candidatos: 368


In [19]:
pares_unicos = {}

for i, j, similitud in pares_candidatos:
    par = tuple(sorted((i, j)))

    if par not in pares_unicos:
        pares_unicos[par] = similitud

print("Pares únicos:", len(pares_unicos))

Pares únicos: 239


## 6. Análisis de candidatos

Los pares encontrados se organizan junto con sus medios, fechas y títulos para facilitar su revisión. Una similitud elevada señala contenido potencialmente repetido, pero no basta para eliminarlo automáticamente: secciones recurrentes pueden mantener estructuras muy parecidas y distintos medios pueden cubrir el mismo hecho con textos legítimamente independientes.


In [20]:
resultados_total = []

for (i, j), similitud in pares_unicos.items():
    resultados_total.append({
        "indice_1": i,
        "indice_2": j,
        "similitud": similitud,
        "medio_1": df_limpio.loc[i, "medio"],
        "fecha_1": df_limpio.loc[i, "fecha"],
        "titulo_1": df_limpio.loc[i, "titulo"],
        "medio_2": df_limpio.loc[j, "medio"],
        "fecha_2": df_limpio.loc[j, "fecha"],
        "titulo_2": df_limpio.loc[j, "titulo"]
    })

df_similares = pd.DataFrame(resultados_total)

df_similares = df_similares.sort_values(
    "similitud",
    ascending=False
).reset_index(drop=True)

df_similares

,indice_1,indice_2,similitud,medio_1,fecha_1,titulo_1,medio_2,fecha_2,titulo_2
0,37333,37446,1.000000,El Peruano,2026-05-18 00:00:00+00:00,Presidente del INPE supervisa seguridad en pen...,El Peruano,2026-05-19 00:00:00+00:00,Presidente del INPE supervisó seguridad en los...
1,7496,7541,0.999418,Perú21,2024-12-23 07:00:00+00:00,Estas son las cortitas de hoy lunes 23 de dici...,Perú21,2024-12-24 07:05:00+00:00,Estas son las cortitas de hoy martes 24 de dic...
2,39229,39230,0.999150,El Comercio,2026-06-07 23:28:07+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...,El Comercio,2026-06-07 23:30:08+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
3,34872,34897,0.999073,El Comercio,2026-04-12 22:14:53.027000+00:00,Elecciones Perú 2026: Resultados de la ONPE de...,El Comercio,2026-04-12 22:23:24.827000+00:00,Elecciones Perú 2026: Resultados de la ONPE de...
4,39220,39230,0.999006,El Comercio,2026-06-07 23:14:04+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...,El Comercio,2026-06-07 23:30:08+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
...,...,...,...,...,...,...,...,...,...
234,1226,1236,0.951710,El Peruano,2024-08-11 00:00:00+00:00,Uno de los legados de mi Gobierno será la mode...,Perú21,2024-08-11 12:10:00+00:00,Dina Boluarte: “Uno de los legados de mi gobie...
235,9634,9684,0.951679,El Peruano,2025-01-31 00:00:00+00:00,Ejecutivo financiará 84 proyectos para cerrar ...,El Peruano,2025-02-01 00:00:00+00:00,Alistan obras en zonas alejadas
236,6976,7016,0.951602,El Peruano,2024-12-15 00:00:00+00:00,Premier Adrianzén supervisó por aire y tierra ...,El Peruano,2024-12-16 00:00:00+00:00,Combaten a mineros ilegales en la frontera con...
237,39910,40027,0.951491,El Peruano,2026-06-17 00:00:00+00:00,Ejecutivo intensifica acciones y coordinacione...,El Peruano,2026-06-18 00:00:00+00:00,Ejecutivo intensifica acciones para enfrentar ...


In [21]:
df_similares["combinacion_medios"] = (
    df_similares["medio_1"] + " - " + df_similares["medio_2"]
)

df_similares["combinacion_medios"].value_counts()

combinacion_medios
El Comercio - El Comercio      118
El Peruano - El Peruano         62
RPP - RPP                       26
Perú21 - Perú21                 12
El Peruano - RPP                 8
RPP - El Peruano                 4
La República - La República      2
Canal N - Canal N                2
El Peruano - Perú21              2
Canal N - El Peruano             1
El Comercio - La República       1
Perú21 - RPP                     1
Name: count, dtype: int64

In [22]:
df_similares["diferencia_horas"] = (
    abs(df_similares["fecha_2"] - df_similares["fecha_1"])
    .dt.total_seconds() / 3600
)

df_similares[
    ["similitud", "medio_1", "medio_2", "diferencia_horas"]
].describe()

,similitud,diferencia_horas
count,239.000000,239.000000
mean,0.980669,94.179294
std,0.015663,358.280444
min,0.951192,0.000000
25%,0.969544,0.162364
50%,0.980779,13.234167
75%,0.998136,24.000000
max,1.000000,3576.000000


## 7. Regla conservadora final

Se consideran candidatos seguros únicamente los pares publicados por el **mismo medio**, con una similitud **mayor o igual a 0.98** y una diferencia temporal **máxima de 6 horas**. La combinación de estos tres requisitos adopta una postura conservadora y reduce el riesgo de descartar noticias distintas que comparten vocabulario, tema o estructura.


In [23]:
# Filtrar candidatos
candidatos = df_similares[
    (df_similares["similitud"] >= 0.98) &
    (df_similares["diferencia_horas"] <= 6) &
    (df_similares["medio_1"] == df_similares["medio_2"])
].copy()

print("Pares candidatos:", len(candidatos))

Pares candidatos: 90


## 8. Agrupación con grafos

Los pares seleccionados se representan como conexiones en un grafo mediante NetworkX. De este modo, si varias versiones están relacionadas entre sí, quedan reunidas en un mismo componente y se tratan como un solo grupo, evitando contar varias veces el mismo caso de repetición.


In [24]:
import networkx as nx

G = nx.Graph()

for _, fila in candidatos.iterrows():
    G.add_edge(
        fila["indice_1"],
        fila["indice_2"]
    )

grupos = list(nx.connected_components(G))

print("Grupos de cuasi-duplicados:", len(grupos))

Grupos de cuasi-duplicados: 12


In [25]:
total_en_grupos = sum(len(grupo) for grupo in grupos)

a_eliminar = sum(
    len(grupo) - 1
    for grupo in grupos
)

print("Noticias involucradas:", total_en_grupos)
print("Noticias a eliminar:", a_eliminar)

Noticias involucradas: 102
Noticias a eliminar: 90


## 9. Eliminación final

Dentro de cada grupo se conserva la noticia con la fecha más antigua y se marcan las demás versiones para su eliminación. Por tanto, esta etapa retira solo los registros clasificados como cuasi-duplicados por la regla anterior y mantiene una noticia representativa de cada grupo.


In [26]:
indices_eliminar = []

for grupo in grupos:
    grupo_lista = list(grupo)

    conservar = df_limpio.loc[
        grupo_lista, "fecha"
    ].idxmin()

    eliminar = [
        i for i in grupo_lista
        if i != conservar
    ]

    indices_eliminar.extend(eliminar)

In [27]:
df_limpio.loc[
    indices_eliminar,
    ["medio", "fecha", "titulo"]
].head(30)

,medio,fecha,titulo
39214,El Comercio,2026-06-07 23:02:00+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39215,El Comercio,2026-06-07 23:04:01+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39216,El Comercio,2026-06-07 23:06:01+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39217,El Comercio,2026-06-07 23:08:02+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39218,El Comercio,2026-06-07 23:10:03+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39219,El Comercio,2026-06-07 23:12:03+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39220,El Comercio,2026-06-07 23:14:04+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39221,El Comercio,2026-06-07 23:16:04+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39222,El Comercio,2026-06-07 23:18:05+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...
39223,El Comercio,2026-06-07 23:20:05+00:00,Elecciones Perú 2026: Aquí podrás ver los Resu...


In [28]:
df_final = df_limpio.drop(index=indices_eliminar).copy()

df_final = df_final.reset_index(drop=True)

print("Antes:", len(df_limpio))
print("Después:", len(df_final))
print("Eliminadas:", len(df_limpio) - len(df_final))

Antes: 41307
Después: 41217
Eliminadas: 90


### Verificación del resultado

La comprobación final contrasta el tamaño del dataset y confirma que no permanezcan textos vacíos ni duplicados exactos. Estas salidas permiten verificar que la depuración fue aplicada de acuerdo con los criterios definidos.


In [29]:
print("Filas finales:", len(df_final))
print("Textos vacíos:", df_final["texto_completo"].isna().sum())
print("Textos duplicados exactos:", df_final.duplicated(subset=["texto_completo"]).sum())

Filas finales: 41217
Textos vacíos: 0
Textos duplicados exactos: 0


## 10. Exportación

El resultado se guarda como `noticias_politica_limpio.csv`. Este archivo constituye la entrada limpia para las siguientes etapas de procesamiento y análisis de lenguaje natural.


In [30]:
df_final.to_csv(
    "noticias_politica_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset guardado correctamente.")

Dataset guardado correctamente.


## 11. Resumen final

El proceso parte de **41,823 noticias originales**. Se eliminan **504 noticias sin texto**, **12 duplicados exactos** y **90 cuasi-duplicados**, por lo que el corpus resultante contiene **41,217 noticias**. Este balance resume el efecto de una limpieza orientada a mejorar la calidad del conjunto sin suprimir contenidos distintos de manera indiscriminada.


In [31]:
print("=== RESUMEN FINAL ===")
print("Noticias originales:", len(df))
print("Sin texto eliminadas:", 504)
print("Duplicados exactos eliminados:", 12)
print("Cuasi-duplicados eliminados:", 90)
print("Noticias finales:", len(df_final))

=== RESUMEN FINAL ===
Noticias originales: 41823
Sin texto eliminadas: 504
Duplicados exactos eliminados: 12
Cuasi-duplicados eliminados: 90
Noticias finales: 41217
